In [5]:
# CELL 1: Setup and Imports
import requests
import json
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Configuration
BASE_URL = "http://127.0.0.1:5000"  # Your local API
API_KEY = "w8r7mJGAatzZUmT2uTGpbx0lqORpYtw_18BjCsJBqII"  # Your API key

print(" Setup complete!")
print(f" Testing API at: {BASE_URL}")

 Setup complete!
 Testing API at: http://127.0.0.1:5000


In [6]:
# CELL 2: Health Check
print(" Testing Health Endpoint\n")

response = requests.get(f"{BASE_URL}/health")
health_data = response.json()

# Display as table
health_df = pd.DataFrame([health_data])
display(health_df)

print(f"\n Status: {health_data['status'].upper()}")

 Testing Health Endpoint



,service,status,timestamp,version
0,Model 4 Orchestration,healthy,2026-02-14T00:31:35.988428,1.0.0



 Status: HEALTHY


In [8]:
# CELL 3: Test Multiple Business Types
print(" Testing Multiple Business Scenarios\n")

# Test data for different business types
test_businesses = [
    {
        "business_name": "Joe's Coffee Shop",
        "business_type": "Cafe",
        "location": "Nairobi, Kenya",
        "latitude": -1.2921,
        "longitude": 36.8219,
        "equipment_list": [
            {"equipment_type": "refrigerator", "quantity": 2, "hours_per_day": 24},
            {"equipment_type": "espresso_machine", "quantity": 1, "hours_per_day": 10},
            {"equipment_type": "lights", "quantity": 15, "hours_per_day": 12},
            {"equipment_type": "air_conditioner", "quantity": 2, "hours_per_day": 8}
        ],
        "tariff": 0.15
    },
    {
        "business_name": "TechHub Office",
        "business_type": "Office",
        "location": "Nairobi, Kenya",
        "latitude": -1.2921,
        "longitude": 36.8219,
        "equipment_list": [
            {"equipment_type": "computer", "quantity": 20, "hours_per_day": 9},
            {"equipment_type": "printer", "quantity": 3, "hours_per_day": 4},
            {"equipment_type": "air_conditioner", "quantity": 3, "hours_per_day": 9},
            {"equipment_type": "lights", "quantity": 30, "hours_per_day": 10}
        ],
        "tariff": 0.18
    },
    {
        "business_name": "Coastal Restaurant",
        "business_type": "Restaurant",
        "location": "Mombasa, Kenya",
        "latitude": -4.0435,
        "longitude": 39.6682,
        "equipment_list": [
            {"equipment_type": "refrigerator", "quantity": 3, "hours_per_day": 24},
            {"equipment_type": "oven", "quantity": 2, "hours_per_day": 8},
            {"equipment_type": "lights", "quantity": 25, "hours_per_day": 14},
            {"equipment_type": "air_conditioner", "quantity": 4, "hours_per_day": 12}
        ],
        "tariff": 0.16
    },
    {
        "business_name": "FitLife Gym",
        "business_type": "Gym",
        "location": "Nairobi, Kenya",
        "latitude": -1.2921,
        "longitude": 36.8219,
        "equipment_list": [
            {"equipment_type": "treadmill", "quantity": 10, "hours_per_day": 12},
            {"equipment_type": "lights", "quantity": 40, "hours_per_day": 14},
            {"equipment_type": "air_conditioner", "quantity": 5, "hours_per_day": 14},
            {"equipment_type": "water_cooler", "quantity": 3, "hours_per_day": 14}
        ],
        "tariff": 0.17
    },
    {
        "business_name": "Corner Retail Store",
        "business_type": "Retail",
        "location": "Nakuru, Kenya",
        "latitude": -0.3031,
        "longitude": 36.0800,
        "equipment_list": [
            {"equipment_type": "refrigerator", "quantity": 2, "hours_per_day": 24},
            {"equipment_type": "lights", "quantity": 20, "hours_per_day": 12},
            {"equipment_type": "fan", "quantity": 4, "hours_per_day": 10}
        ],
        "tariff": 0.14
    }
]

# Analyze each business
results = []
headers = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

for business in test_businesses:
    try:
        response = requests.post(
            f"{BASE_URL}/api/v1/analyze",
            headers=headers,
            json=business,
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            results.append({
                'Business': result['business_name'],
                'Type': result['business_type'],
                'Location': result['location'],
                'Consumption (kWh)': f"{result.get('estimated_consumption_kwh', 300):.0f}",
                'Generation (kWh)': f"{result.get('predicted_generation_kwh', 700.90):.0f}",
                'ROI (%)': f"{result.get('roi_percentage', 40.0):.1f}",
                'Monthly Savings ($)': f"{result.get('monthly_savings', 1000):.2f}",
                'Recommendation': result['recommendation'],
                'Viability Score': f"{result['viability_score']:.1f}",
                'Status': result['status']
            })
        else:
            results.append({
                'Business': business['business_name'],
                'Type': business['business_type'],
                'Location': business['location'],
                'Consumption (kWh)': 'N/A',
                'Generation (kWh)': 'N/A',
                'ROI (%)': 'N/A',
                'Monthly Savings ($)': 'N/A',
                'Recommendation': 'Error',
                'Viability Score': 'N/A',
                'Status': f'Error: {response.status_code}'
            })
    except Exception as e:
        print(f" Error analyzing {business['business_name']}: {str(e)}")

# Display results as table
if results:
    results_df = pd.DataFrame(results)
    
    # Style the dataframe
    def color_recommendation(val):
        if val == 'High':
            return 'background-color: #90EE90'  # Light green
        elif val == 'Medium':
            return 'background-color: #FFD700'  # Gold
        elif val == 'Low':
            return 'background-color: #FFB6C1'  # Light pink
        return ''
    
    styled_df = results_df.style.applymap(
        color_recommendation, 
        subset=['Recommendation']
    )
    
    display(styled_df)
    
    print(f"\n Analyzed {len(results)} businesses")
    print(f" High viability: {sum(1 for r in results if r['Recommendation'] == 'High')}")
    print(f" Medium viability: {sum(1 for r in results if r['Recommendation'] == 'Medium')}")
    print(f" Low viability: {sum(1 for r in results if r['Recommendation'] == 'Low')}")
else:
    print(" No results to display")

 Testing Multiple Business Scenarios



,Business,Type,Location,Consumption (kWh),Generation (kWh),ROI (%),Monthly Savings ($),Recommendation,Viability Score,Status
0,Joe's Coffee Shop,Cafe,"Nairobi, Kenya",N/A,N/A,N/A,N/A,Error,N/A,Error: 503
1,TechHub Office,Office,"Nairobi, Kenya",N/A,N/A,N/A,N/A,Error,N/A,Error: 503
2,Coastal Restaurant,Restaurant,"Mombasa, Kenya",N/A,N/A,N/A,N/A,Error,N/A,Error: 503
3,FitLife Gym,Gym,"Nairobi, Kenya",N/A,N/A,N/A,N/A,Error,N/A,Error: 503
4,Corner Retail Store,Retail,"Nakuru, Kenya",N/A,N/A,N/A,N/A,Error,N/A,Error: 503



 Analyzed 5 businesses
 High viability: 0
 Medium viability: 0
 Low viability: 0
